<!-- nav -->
[← 2 · One value, a domain in each language](02_One_Value_A_Domain_In_Each_Language.ipynb) · [Home](../../README.md) · [4 · Names →](04_Names.ipynb)

# 3 · The smallest expression, a literal

Tutorial 2 read, by name, the domain each language's standard gives `42`. This one writes `42` as an expression, the
smallest there is: one literal. In every dialect a literal is a **term**, a value, held by an **expression**, a
reference object of the store; each language prints it as its own text, and Python's is read back from that text. Its
domain is inferred and its value evaluated, each by its language's rules. Then come the literals no scalar holds, such
as Python's `2j` and `None`, and a literal whose text holds no value, kept as written.

Every model here is written in the builders' fluent calls
([PRI-019: fluent builder syntax](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-019)), which show the model as the store holds it; the writers
Part II uses are shown once, in 03.1, as the shorthand they are. Part I starts from the toolkit's `dialects()`, a store
with every dialect installed and defined. `S` is its holder namespace, `store.Expressions`, and `amd64` x86-64's System V
ABI, the dialect C's terms are read under (02.3).

In [1]:
from mbse.Expressions import Translators
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.Framework import Terms
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Python import Python3
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.TypeScript import TypeScript5
from toolkit import dialects

store = dialects()
S = store.Expressions
amd64 = store.Instances.Expressions.SystemV.AMD64

<a id="T03.1"></a>
## 03.1 · A term, held by an expression

An expression is a reference object, made in a transaction by its dialect's **holder**: Patterns' is
`S.Patterns.Expressions.Expression`. Its `root` holds a term, a value of the dialect's `Term` union, whose branch is its
kind, here `literal`, with its `value`, `42`. `forty_two` is that term's spec, the callable the builders call with the
union to write, so it writes `42` into any dialect's `Term` with a `literal` kind. `Terms.term` reads the term an
expression holds, `Terms.dialect_of` and `Terms.kind_of` its dialect and kind, and `Terms.form` its kind, its attributes
and its arguments, of which a literal has none:

In [2]:
forty_two = lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(42)))  # a spec: the term 42, in any dialect's Term
with store.transaction():
    answer = S.Patterns.Expressions.Expression(lambda e: e.root(forty_two))
term = Terms.term(answer)
form = Terms.form(term)
print(answer.as_schema().name, "ref:", answer.as_schema().ref)
print(Terms.dialect_of(term), Terms.kind_of(term)[0])
print(form.kind, form.attributes, len(form.arguments))
assert Terms.holder_of(term) == answer
assert (form.kind, form.attributes, form.arguments) == ("literal", {"value": 42}, ())

Expressions.Patterns.Expressions.Expression ref: True
Expressions.Patterns.Expressions.Term literal
literal {'value': 42} 0


Part II writes its terms with **writers** instead: each dialect's functions that write a term through these same
builders, a shorthand for the calls above. A writer writes the same term as the builder calls, which `Terms.same`
compares by structure (dialect, kind, attributes and arguments). Here are Patterns', Python3's, C's and SystemVerilog's:

In [3]:
with store.transaction():
    pairs = {
        "Patterns": (S.Patterns.Expressions.Expression(lambda e: e.root(E.literal(42))), answer),
        "Python3": (S.Python.Python3.Expression(lambda e: e.root(Python3.Expressions.constant(42))),
                    S.Python.Python3.Expression(lambda e: e.root(forty_two))),
        "C": (S.C.Expressions.Expression(lambda e: e.root(C.Expressions.literal(42)).abi([amd64])),
              S.C.Expressions.Expression(lambda e: e.root(forty_two).abi([amd64]))),
        "SystemVerilog": (S.SystemVerilog.Expressions.Expression(lambda e: e.root(SystemVerilog.Expressions.literal(42))),
                          S.SystemVerilog.Expressions.Expression(lambda e: e.root(forty_two))),
    }
for dialect, (written, built) in pairs.items():
    print(f"{dialect:13}", Terms.same(written, built))
    assert Terms.same(written, built)

Patterns      True
Python3       True
C             True
SystemVerilog True


The rest of Part I writes builder calls, so that what you read is the model itself; Part II keeps its writers.

<a id="T03.2"></a>
## 03.2 · Read and printed

Python3's `42` is the same spec in its own holder, `S.Python.Python3.Expression`, and `Python3.Text.ToText` prints it as
Python source. A literal also keeps how it was spelled, beside its value: `0x2A` is the value 42, with its `text` as
written and the `base` that text encodes, 16, so it prints back as written
([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)). Python reads Python source too:
`Python3.Text.FromText` gives the writer of the term a text writes, and `0x2A`, read, is the term built here. Only Python
reads it, with Python's own parser; TypeScript builds the same term
([Equivalence of the implementations](../../docs/EQUIVALENCE.md#deliberate-differences), "Reading Python source").

In [4]:
with store.transaction():
    plain = S.Python.Python3.Expression(lambda e: e.root(forty_two))
    hexadecimal = S.Python.Python3.Expression(lambda e: e.root(
        lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(42)).text("0x2A").base(16))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("0x2A")))  # Python reads it
for expression in (plain, hexadecimal):
    print(f"{Python3.Text.ToText(expression):5}", Terms.form(Terms.term(expression)).attributes)
assert [Python3.Text.ToText(x) for x in (plain, hexadecimal)] == ["42", "0x2A"]
assert Terms.same(read, hexadecimal)

42    {'value': 42}
0x2A  {'value': 42, 'text': '0x2A', 'base': 16}


<a id="T03.3"></a>
## 03.3 · The same literal in each language

Each language's dialect has its own holder and its own `Term`, whose `literal` kind refines the framework's, with what
its language's spelling encodes. TypeScript's `42` is a float, as every JavaScript number is: its value is `42.0`, its
`text` `42`. C's holder names the ABI its terms are read under, `amd64`. Each prints its own text, Excel's a formula:

In [5]:
with store.transaction():
    literals = {
        "TypeScript5": (S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(42.0)).text("42")))), TypeScript5.Text),
        "C": (S.C.Expressions.Expression(lambda e: e.root(forty_two).abi([amd64])), C.Text),
        "SystemVerilog": (S.SystemVerilog.Expressions.Expression(lambda e: e.root(forty_two)), SystemVerilog.Text),
        "Excel": (S.Excel.Expressions.Expression(lambda e: e.root(forty_two)), Excel.Text),
        "MATLAB": (S.MATLAB.Expressions.Expression(lambda e: e.root(forty_two)), Matlab.Text),
        "LaTeX": (S.LaTeX.Expressions.Expression(lambda e: e.root(forty_two)), Latex.Text),
    }
for dialect, (expression, text) in literals.items():
    print(f"{dialect:13} {text.ToText(expression):4}", Terms.dialect_of(Terms.term(expression)))
assert [text.ToText(expression) for expression, text in literals.values()] == ["42", "42", "42", "=42", "42", "42"]

TypeScript5   42   Expressions.TypeScript.TypeScript5.Term
C             42   Expressions.C.Expressions.Term
SystemVerilog 42   Expressions.SystemVerilog.Expressions.Term
Excel         =42  Expressions.Excel.Expressions.Term
MATLAB        42   Expressions.MATLAB.Expressions.Term
LaTeX         42   Expressions.LaTeX.Expressions.Term


Each term holds its dialect, so none is another's: Python3's `42` and C's are not the same term, and C's dialect reads
no Python3 term:

In [6]:
c = literals["C"][0]
print(Terms.same(plain, c), C.Expressions.DIALECT.validate(plain))
assert not Terms.same(plain, c) and C.Expressions.DIALECT.validate(plain) == ["a Python3 literal is not a C term"]

False ['a Python3 literal is not a C term']


<a id="T03.4"></a>
## 03.4 · Its domain, inferred

Each dialect's `DIALECT.infer` gives the domain of a term's value: the domains tutorial 2 read by name, Python's `int`,
TypeScript's `number`, C's `int` under `amd64` and SystemVerilog's `integer`, 4-state (an unsized decimal literal is an
`integer`, not an `int`), and Excel's `number` and MATLAB's `double`. Patterns has no stored domains, by design: its
`int` is a native type's. C's `int` is the ABI's, not C's, so a C literal in a holder that names no ABI has no domain.
Inference is code until [CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md) makes it data:

In [7]:
inferred = {
    "Patterns": (E.DIALECT, answer),
    "Python3": (Python3.Expressions.DIALECT, plain),
    "TypeScript5": (TypeScript5.Expressions.DIALECT, literals["TypeScript5"][0]),
    "C": (C.Expressions.DIALECT, c),
    "SystemVerilog": (SystemVerilog.Expressions.DIALECT, literals["SystemVerilog"][0]),
    "Excel": (Excel.Expressions.DIALECT, literals["Excel"][0]),
    "MATLAB": (Matlab.Expressions.DIALECT, literals["MATLAB"][0]),
}
for dialect, (reader, expression) in inferred.items():
    print(f"{dialect:13}", reader.infer(expression).name())
assert [reader.infer(x).name() for reader, x in inferred.values()] == [
    "int", "int", "number", "int", "integer", "number", "double"]
with store.transaction():
    no_abi = S.C.Expressions.Expression(lambda e: e.root(forty_two))
try:
    C.Expressions.DIALECT.infer(no_abi)
    raise AssertionError("inferred")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Patterns      int
Python3       int
TypeScript5   number
C             int
SystemVerilog integer
Excel         number
MATLAB        double
ValueError: cannot infer the domain of an invalid expression: 'int' is not a type C defines, and its expression names no ABI


<a id="T03.5"></a>
## 03.5 · Its value

Each dialect's `Evaluators.OfAny` evaluates a term by its language's rules: Patterns', Python's and C's `42` is an
integer; TypeScript's a float, as are Excel's and MATLAB's, whose numbers are doubles; SystemVerilog's a 4-state value of
its domain, `integer`, 32 bits signed, written as SystemVerilog writes it. LaTeX typesets, and evaluates nothing: it has
no evaluator.

In [8]:
evaluated = {
    "Patterns": (Evaluators, answer),
    "Python3": (Python3.Evaluators, plain),
    "TypeScript5": (TypeScript5.Evaluators, literals["TypeScript5"][0]),
    "C": (C.Evaluators, c),
    "SystemVerilog": (SystemVerilog.Evaluators, literals["SystemVerilog"][0]),
    "Excel": (Excel.Evaluators, literals["Excel"][0]),
    "MATLAB": (Matlab.Evaluators, literals["MATLAB"][0]),
}
for dialect, (evaluators, expression) in evaluated.items():
    print(f"{dialect:13}", evaluators.OfAny(expression))
print("LaTeX has an evaluator:", hasattr(Latex, "Evaluators"))
assert [str(evaluators.OfAny(x)) for evaluators, x in evaluated.values()] == [
    "42", "42", "42.0", "42", "32'sb00000000000000000000000000101010", "42.0", "42.0"]

Patterns      42
Python3       42
TypeScript5   42.0
C             42
SystemVerilog 32'sb00000000000000000000000000101010
Excel         42.0
MATLAB        42.0
LaTeX has an evaluator: False


<a id="T03.6"></a>
## 03.6 · 2.5, a string, a boolean

Other scalars are literals the same way. `Translators.between(source, target).forward(term)` translates a term to
another dialect (tutorial 14 tells how); here Patterns' `2.5`, `'hi'` and `True` in each language, a boolean spelled
six ways:

In [9]:
Term = S.Framework.Term
targets = {"Python3": (Python3.Expressions.DIALECT, Python3.Text), "C": (C.Expressions.DIALECT, C.Text),
           "SystemVerilog": (SystemVerilog.Expressions.DIALECT, SystemVerilog.Text),
           "Excel": (Excel.Expressions.DIALECT, Excel.Text), "MATLAB": (Matlab.Expressions.DIALECT, Matlab.Text),
           "LaTeX": (Latex.Expressions.DIALECT, Latex.Text)}
print("".join(f"{dialect:16}" for dialect in targets).rstrip())
for value in (2.5, "hi", True):
    with store.transaction():
        patterns = S.Patterns.Expressions.Expression(lambda e: e.root(
            lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(value)))))
        translated = [Term(Translators.between(E.DIALECT, dialect).forward(patterns)) for dialect, _ in targets.values()]
    print("".join(f"{text.ToText(t):16}" for t, (_, text) in zip(translated, targets.values())).rstrip())
assert [text.ToText(t) for t, (_, text) in zip(translated, targets.values())] == [
    "True", "true", "1'b1", "=TRUE", "true", r"\mathrm{true}"]

Python3         C               SystemVerilog   Excel           MATLAB          LaTeX
2.5             2.5             2.5             =2.5            2.5             2.5
'hi'            "hi"            "hi"            ="hi"           "hi"            \text{hi}
True            true            1'b1            =TRUE           true            \mathrm{true}


A string's quotes are part of its spelling: Python's `'hi'` and `"hi"`, read or built, are the same value but not the same
term, since `Terms.same` compares their `text` and `quote` too:

In [10]:
with store.transaction():
    single = S.Python.Python3.Expression(lambda e: e.root(
        lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar("hi")).text("'hi'").quote("'"))))
    double = S.Python.Python3.Expression(lambda e: e.root(
        lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar("hi")).text('"hi"').quote('"'))))
    read = [S.Python.Python3.Expression(lambda e, source=source: e.root(Python3.Text.FromText(source)))
            for source in ("'hi'", '"hi"')]  # Python reads them
print(Python3.Text.ToText(single), Python3.Text.ToText(double), "same term:", Terms.same(single, double),
      "same value:", Python3.Evaluators.OfAny(single) == Python3.Evaluators.OfAny(double))
assert not Terms.same(single, double) and Python3.Evaluators.OfAny(single) == Python3.Evaluators.OfAny(double) == "hi"
assert Terms.same(read[0], single) and Terms.same(read[1], double)

'hi' "hi" same term: False same value: True


<a id="T03.7"></a>
## 03.7 · What no scalar holds

A literal holds a value of a schema
([CAP-233: literals of any value](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-233.md)): a scalar, a container or a value object. What has
no schema is its dialect's own kind, `constant`, with its spelling too: Python's `2j`, a `complex` of two IEEE 754
`binary64` parts, and the singletons `None` and `...`; TypeScript's `null` and a regular expression, its `pattern` and
`flags`; C's `nullptr`. Each prints back as written, and
Python reads Python's three, with their spelling, as the terms built here:

In [11]:
with store.transaction():
    constants = {
        "2j": (S.Python.Python3.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.complex(lambda z: z.real(0.0).imag(2.0)).text("2j")))), Python3.Text),
        "None": (S.Python.Python3.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.singleton("None").text("None")))), Python3.Text),
        "...": (S.Python.Python3.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.singleton("...").text("...")))), Python3.Text),
        "null": (S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.singleton("null")))), TypeScript5.Text),
        "/a+/gu": (S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.pattern("a+").flags("gu")))), TypeScript5.Text),
        "nullptr": (S.C.Expressions.Expression(lambda e: e.root(
            lambda u: u.constant(lambda k: k.singleton("nullptr"))).abi([amd64])), C.Text),
    }
for expression, text in constants.values():
    form = Terms.form(Terms.term(expression))
    print(f"{text.ToText(expression):8} {Terms.dialect_of(Terms.term(expression)):39}", form.kind, list(form.attributes))
assert [text.ToText(expression) for expression, text in constants.values()] == list(constants)
with store.transaction():  # Python reads them
    read = {source: S.Python.Python3.Expression(lambda e, source=source: e.root(Python3.Text.FromText(source)))
            for source in ("2j", "None", "...")}
for source, expression in read.items():
    print(f"{source:4}", Terms.same(expression, constants[source][0]))
    assert Terms.same(expression, constants[source][0])

2j       Expressions.Python.Python3.Term         constant ['complex', 'text']
None     Expressions.Python.Python3.Term         constant ['singleton', 'text']
...      Expressions.Python.Python3.Term         constant ['singleton', 'text']
null     Expressions.TypeScript.TypeScript5.Term constant ['singleton']
/a+/gu   Expressions.TypeScript.TypeScript5.Term constant ['pattern', 'flags']
nullptr  Expressions.C.Expressions.Term          constant ['singleton']
2j   True
None True
...  True


A `constant`'s domain is inferred, `complex`, `NoneType` or `ellipsis`, Python's names for its type, but Python3's
evaluator refuses to evaluate one, by name:

In [12]:
for source in ("2j", "None", "..."):
    expression = constants[source][0]
    try:
        Python3.Evaluators.OfAny(expression)
        raise AssertionError("evaluated")
    except NotImplementedError as error:
        print(f"{source:4} {Python3.Expressions.DIALECT.infer(expression).name():8} {type(error).__name__}: {error}")

2j   complex  NotImplementedError: Python3 constant is not evaluated
None NoneType NotImplementedError: Python3 constant is not evaluated
...  ellipsis NotImplementedError: Python3 constant is not evaluated


<a id="T03.8"></a>
## 03.8 · A literal kept as written, valid or not

A literal's text may hold no value: `08` is no Python int, since Python 3 forbids leading zeros in a decimal literal. A
reader that keeps what it reads ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)) writes it as a literal with
its `text` and its `problems`, and no `value`. It prints back as written; validation reports its problems, and
inference and evaluation refuse it. Python's own parser refuses `08`, so `FromText` cannot read it: the model is built.

In [13]:
with store.transaction():
    invalid = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.literal(
        lambda r: r.text("08").problems(["leading zeros in a decimal integer literal are not permitted"]))))
print(Python3.Text.ToText(invalid), list(Terms.form(Terms.term(invalid)).attributes))
print(Python3.Expressions.DIALECT.validate(invalid))
for refused in (Python3.Expressions.DIALECT.infer, Python3.Evaluators.OfAny):
    try:
        refused(invalid)
        raise AssertionError("not refused")
    except ValueError as error:
        print(f"{type(error).__name__}: {error}")
assert Python3.Text.ToText(invalid) == "08"
assert Python3.Expressions.DIALECT.validate(invalid) == ["leading zeros in a decimal integer literal are not permitted"]
try:  # Python's own parser refuses it: FromText cannot read it
    Python3.Text.FromText("08")
    raise AssertionError("read")
except SyntaxError as error:
    assert "leading zeros" in str(error)

08 ['text', 'problems']
['leading zeros in a decimal integer literal are not permitted']
ValueError: cannot infer the domain of an invalid expression: leading zeros in a decimal integer literal are not permitted
ValueError: a literal needs a value


## Why it works this way

- **A term is a value, its expression a reference object.** A literal is data its holder owns, compared by structure
  (`Terms.same`), never by identity; the expression gives it a place in the store, its dialect and, for C, its ABI.
- **A literal keeps its spelling beside its value**
  ([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)), so that text printed from a model is the text it
  was read from, and a text that holds no value is kept, with its problems
  ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)).
- **What no schema holds is a dialect's own kind** ([CAP-233: literals of any value](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-233.md)):
  Python's complex numbers and singletons, TypeScript's `null` and regular expressions, C's `nullptr`.
- **A domain is inferred, not stored on the term**, until
  [CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md); tutorial 4 goes on to names.

<!-- nav -->
[← 2 · One value, a domain in each language](02_One_Value_A_Domain_In_Each_Language.ipynb) · [Home](../../README.md) · [4 · Names →](04_Names.ipynb)